# CSE 590 Programming Assignment 5: Speculative Decoding

## Setup

In [1]:
import os
import torch
import time
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from typing import List, Tuple, Dict, Optional

## Speculative Decoding

In [7]:
class SpeculativeDecoder:
    def __init__(self, target_model_name: str, draft_model_name: str, device: str = "cuda"):
        """
        Initialize the speculative decoder with target and draft models.

        Args:
            target_model_name: HuggingFace model ID for the larger target model.
            draft_model_name: HuggingFace model ID for the smaller draft model.
            device: Device to run models on ("cuda" or "cpu").
        """
        self.device = device
        self.target_model, self.target_tokenizer = self.initialize_target_model(target_model_name)
        self.draft_model, self.draft_tokenizer = self.initialize_draft_model(draft_model_name)

        # Ensure tokenizers are compatible
        assert self.target_tokenizer.vocab == self.draft_tokenizer.vocab, "Tokenizers must be compatible"

    def initialize_target_model(self, model_name: str):
        """Initialize the larger target model with caching enabled and proper pad token."""
        print(f"Loading target model: {model_name}")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if "cuda" in self.device else torch.float32,
            low_cpu_mem_usage=True,
        )
        model.to(self.device)
        model.eval()
        model.config.pad_token_id = tokenizer.pad_token_id
        return model, tokenizer

    def initialize_draft_model(self, model_name: str):
        """
        Initialize a smaller, faster draft model with proper pad token.
        """
        print(f"Loading draft model: {model_name}")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if "cuda" in self.device else torch.float32,
            low_cpu_mem_usage=True,
        )
        model.to(self.device)
        model.eval()
        model.config.pad_token_id = tokenizer.pad_token_id
        return model, tokenizer

    def generate_draft_tokens(self, input_ids: torch.Tensor, attention_mask: torch.Tensor,
                             num_speculative_tokens: int = 10) -> torch.Tensor:
        """
        Generate speculative tokens in one forward call using the draft model.

        Args:
            input_ids: Input token IDs (tensor of shape [1, seq_len]).
            attention_mask: Corresponding attention mask.
            num_speculative_tokens: Number of tokens to speculate.

        Returns:
            Tensor of shape [1, num_speculative_tokens] containing the draft tokens.
        """
        # Hint: Please use greedy decoding
        with torch.no_grad():
            output_ids = self.draft_model.generate(
                input_ids,
                attention_mask=attention_mask,
                max_new_tokens=num_speculative_tokens,
                do_sample=False,
                pad_token_id=self.draft_tokenizer.pad_token_id,
            )
        generated_tokens = output_ids[:, input_ids.shape[1]:]
        return generated_tokens

    def verify_tokens_vectorized(self, input_ids: torch.Tensor, draft_tokens: torch.Tensor,
                               attention_mask: torch.Tensor) -> Tuple[List[int], int]:
        """
        Vectorized verification: verify all draft tokens in one forward pass using the target model.

        Args:
            input_ids: The current input token IDs (shape [1, L]).
            draft_tokens: Draft tokens from the draft model (shape [1, k]).
            attention_mask: The current attention mask for input_ids.

        Returns:
            accepted_tokens: List of accepted token IDs.
            accepted_position: Index of the first rejected token (if all accepted, equals draft_tokens.shape[1]).
            last_logit: the logit of the first token after the last accepted token. Used for TODO 4 inside speculative_decode function
        """
        k = draft_tokens.shape[1]

        if k == 0:
            with torch.no_grad():
                outputs = self.target_model(input_ids=input_ids, attention_mask=attention_mask)
            return [], 0, outputs.logits[:, -1, :]

        # 1. Run target model on input_ids concatenated with draft_tokens
        combined_ids = torch.cat([input_ids, draft_tokens], dim=1)
        combined_mask = torch.cat(
            [attention_mask, attention_mask.new_ones((1, k))], dim=1
        )
        with torch.no_grad():
            outputs = self.target_model(input_ids=combined_ids, attention_mask=combined_mask)

        # 2. Extract logits for positions predicting draft tokens
        L = input_ids.shape[1]
        target_preds = torch.argmax(outputs.logits[:, L - 1 : L - 1 + k, :], dim=-1)

        # 3. Compare target model predictions with draft tokens
        matches = target_preds[0] == draft_tokens[0]

        # 4. Find first mismatch
        mismatch_positions = (~matches).nonzero(as_tuple=False)
        if mismatch_positions.numel() == 0:
            accepted_position = k
        else:
            accepted_position = int(mismatch_positions[0].item())

        accepted_tokens = draft_tokens[0, :accepted_position].tolist()
        last_logit = outputs.logits[:, L - 1 + accepted_position, :]

        return accepted_tokens, accepted_position, last_logit

    def speculative_decode(self, prompt: str, max_tokens: int = 100,
                          num_speculative_tokens: int = 15,
                          return_ids: bool = False):
        """
        Main speculative decoding algorithm with vectorized verification.

        Args:
            prompt: Input text.
            max_tokens: Maximum number of tokens to generate (excluding prompt).
            num_speculative_tokens: Number of tokens to speculate per iteration.
            return_ids: If True, return the generated token ID tensor instead of decoded text.

        Returns:
            Generated text (str), or generated token IDs (Tensor) when return_ids=True.
        """
        # Tokenize prompt
        inputs = self.target_tokenizer(prompt, return_tensors="pt", padding=True)
        input_ids = inputs["input_ids"].to(self.device)
        attention_mask = inputs["attention_mask"].to(self.device)
        prompt_length = input_ids.shape[1]

        # Initialize counters for performance tracking
        total_tokens_generated = prompt_length
        total_draft_tokens_proposed = 0
        total_draft_tokens_accepted = 0
        start_time = time.time()

        generated_count = 0
        eos_id = self.target_tokenizer.eos_token_id

        # Pre-allocate output buffer to avoid repeated torch.cat allocations.
        # Max possible length: prompt + max_tokens + one extra for overshoot safety.
        max_len = prompt_length + max_tokens + num_speculative_tokens + 1
        buf = torch.zeros((1, max_len), dtype=input_ids.dtype, device=self.device)
        buf[0, :prompt_length] = input_ids[0]
        seq_len = prompt_length  # current filled length in buffer

        with torch.inference_mode():
            while generated_count < max_tokens:
                remaining = max_tokens - generated_count
                k = min(num_speculative_tokens, remaining)

                # Current sequence view (no copy -- shares memory with buf)
                cur_ids = buf[:, :seq_len]

                # ── 1. Fast draft: prefill + cached autoregressive loop ──
                # Bypasses HuggingFace generate() overhead (~20ms Python setup
                # per call) by calling model() directly with use_cache=True.
                d_out = self.draft_model(input_ids=cur_ids, use_cache=True)
                d_cache = d_out.past_key_values
                tok = d_out.logits[:, -1:, :].argmax(dim=-1)
                draft_list = [tok]
                for _ in range(k - 1):
                    d_out = self.draft_model(
                        input_ids=tok, past_key_values=d_cache, use_cache=True
                    )
                    d_cache = d_out.past_key_values
                    tok = d_out.logits[:, -1:, :].argmax(dim=-1)
                    draft_list.append(tok)
                draft_tokens = torch.cat(draft_list, dim=1)  # [1, k]
                n_draft = draft_tokens.shape[1]
                total_draft_tokens_proposed += n_draft

                # ── 2. Verify: single cache-free target forward pass ─────
                # Build combined sequence in buffer to avoid allocation
                buf[0, seq_len:seq_len + n_draft] = draft_tokens[0]
                combined_ids = buf[:, :seq_len + n_draft]

                t_out = self.target_model(input_ids=combined_ids)

                L = seq_len
                target_preds = t_out.logits[0, L - 1 : L - 1 + n_draft, :].argmax(dim=-1)
                matches = target_preds == draft_tokens[0]
                mismatch = (~matches).nonzero(as_tuple=False)
                accepted = int(n_draft if mismatch.numel() == 0 else mismatch[0].item())
                total_draft_tokens_accepted += accepted

                # ── 3. Accept verified prefix ────────────────────────────
                if accepted > 0:
                    # Draft tokens are already in buf at positions seq_len..seq_len+accepted-1
                    seq_len += accepted
                    generated_count += accepted
                    total_tokens_generated += accepted

                    if eos_id is not None and (draft_tokens[0, :accepted] == eos_id).any():
                        break

                if generated_count >= max_tokens:
                    break

                # ── 4. Correction token from target logits ───────────────
                next_tok_id = t_out.logits[0, L - 1 + accepted, :].argmax(dim=-1)
                buf[0, seq_len] = next_tok_id
                seq_len += 1
                generated_count += 1
                total_tokens_generated += 1

                if eos_id is not None and next_tok_id.item() == eos_id:
                    break

        # Reconstruct final input_ids from buffer for return
        input_ids = buf[:, :seq_len]

        # Calculate performance metrics
        elapsed_time = time.time() - start_time
        acceptance_rate = total_draft_tokens_accepted / total_draft_tokens_proposed if total_draft_tokens_proposed > 0 else 0

        print(f"Generated {total_tokens_generated - prompt_length} tokens in {elapsed_time:.2f} seconds")
        print(f"Tokens per second: {(total_tokens_generated - prompt_length) / elapsed_time:.2f}")
        print(f"Draft token acceptance rate: {acceptance_rate:.2%}")

        if return_ids:
            return input_ids[0, prompt_length:]
        return self.target_tokenizer.decode(input_ids[0], skip_special_tokens=True)

    def benchmark(self, prompt: str, max_tokens: int = 100,
                  num_runs: int = 3, compare_baseline: bool = True) -> Dict:
        """
        Benchmark the speculative decoder against baseline decoding.

        Args:
            prompt: Input text.
            max_tokens: Maximum number of tokens to generate.
            num_runs: Number of benchmark runs.
            compare_baseline: Whether to compare with baseline (non-speculative) decoding.

        Returns:
            Dictionary with benchmark results.
        """
        results = {
            "speculative": {"times": [], "tokens_per_second": []},
            "baseline": {"times": [], "tokens_per_second": []} if compare_baseline else None
        }

        # Benchmark speculative decoding.
        for _ in range(num_runs):
            start_time = time.time()
            output = self.speculative_decode(prompt, max_tokens=max_tokens)
            elapsed = time.time() - start_time
            prompt_len = len(self.target_tokenizer(prompt)["input_ids"])
            output_tokens = len(self.target_tokenizer.encode(output)) - prompt_len
            tps = output_tokens / elapsed
            results["speculative"]["times"].append(elapsed)
            results["speculative"]["tokens_per_second"].append(tps)

        # Benchmark baseline decoding.
        if compare_baseline:
            for _ in range(num_runs):
                inputs = self.target_tokenizer(prompt, return_tensors="pt", padding=True)
                input_ids = inputs["input_ids"].to(self.device)
                attention_mask = inputs["attention_mask"].to(self.device)
                start_time = time.time()
                with torch.no_grad():
                    output_ids = self.target_model.generate(
                        input_ids,
                        attention_mask=attention_mask,
                        max_length=input_ids.shape[1] + max_tokens,
                        do_sample=False,
                        pad_token_id=self.target_tokenizer.pad_token_id
                    )
                elapsed = time.time() - start_time
                output_tokens = output_ids.shape[1] - input_ids.shape[1]
                tps = output_tokens / elapsed
                results["baseline"]["times"].append(elapsed)
                results["baseline"]["tokens_per_second"].append(tps)

        for method in results.keys():
            if results[method] is not None:
                avg_time = sum(results[method]["times"]) / num_runs
                avg_tps = sum(results[method]["tokens_per_second"]) / num_runs
                results[method]["avg_time"] = avg_time
                results[method]["avg_tokens_per_second"] = avg_tps

        if compare_baseline:
            speedup = results["baseline"]["avg_time"] / results["speculative"]["avg_time"]
            results["speedup"] = speedup
            results["latency_reduction"] = (1 - results["speculative"]["avg_time"] / results["baseline"]["avg_time"]) * 100

        return results


## Performance Test

In [8]:
target_model_name = "EleutherAI/pythia-1.4b-deduped"  # Larger target model
draft_model_name = "EleutherAI/pythia-160m-deduped"   # Smaller draft model


# Initialize speculative decoder
decoder = SpeculativeDecoder(
    target_model_name=target_model_name,
    draft_model_name=draft_model_name,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

# Test prompts
test_prompts = [
    "The future of Artificial Intelligence is",
    "Write a short story about a robot learning to feel emotions:",
    "Write the lyrics to the song 'Happy Birthday'."
]

# Run benchmark on test prompts
for i, prompt in enumerate(test_prompts):
    print(f"\nBenchmarking Prompt {i+1}:")
    print(f"Prompt: {prompt}")

    results = decoder.benchmark(
        prompt=prompt,
        max_tokens=100,
        num_runs=3,
        compare_baseline=True
    )

    print(f"Average speculative decoding time: {results['speculative']['avg_time']:.2f} seconds")
    print(f"Average speculative tokens per second: {results['speculative']['avg_tokens_per_second']:.2f}")

    if results["baseline"] is not None:
        print(f"Average baseline decoding time: {results['baseline']['avg_time']:.2f} seconds")
        print(f"Average baseline tokens per second: {results['baseline']['avg_tokens_per_second']:.2f}")
        print(f"Speedup: {results['speedup']:.2f}x")
        print(f"Latency reduction: {results['latency_reduction']:.2f}%")


Loading target model: EleutherAI/pythia-1.4b-deduped


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Loading draft model: EleutherAI/pythia-160m-deduped


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Benchmarking Prompt 1:
Prompt: The future of Artificial Intelligence is
Generated 100 tokens in 1.19 seconds
Tokens per second: 84.16
Draft token acceptance rate: 86.11%
Generated 100 tokens in 1.42 seconds
Tokens per second: 70.53
Draft token acceptance rate: 86.11%
Generated 100 tokens in 1.44 seconds
Tokens per second: 69.49
Draft token acceptance rate: 86.11%
Average speculative decoding time: 1.35 seconds
Average speculative tokens per second: 74.64
Average baseline decoding time: 2.05 seconds
Average baseline tokens per second: 48.90
Speedup: 1.52x
Latency reduction: 34.00%

Benchmarking Prompt 2:
Prompt: Write a short story about a robot learning to feel emotions:
Generated 100 tokens in 1.03 seconds
Tokens per second: 97.02
Draft token acceptance rate: 93.07%
Generated 100 tokens in 1.02 seconds
Tokens per second: 98.41
Draft token acceptance rate: 93.07%
Generated 100 tokens in 1.02 seconds
Tokens per second: 98.47
Draft token acceptance rate: 93.07%
Average speculative decod

## Correctness Check
Speculative decoding is a lossless acceleration technique — it must produce **identical** token sequences to direct autoregressive generation from the target model alone.

In [9]:
print("=== Correctness Check: Speculative Decoding vs. Direct Target Generation ===\n")

max_tokens_check = 50
all_passed = True

for i, prompt in enumerate(test_prompts):
    print(f"Prompt {i+1}: {prompt!r}")

    spec_ids = decoder.speculative_decode(prompt, max_tokens=max_tokens_check, return_ids=True)

    inputs = decoder.target_tokenizer(prompt, return_tensors="pt", padding=True)
    input_ids_direct = inputs["input_ids"].to(decoder.device)
    attention_mask_direct = inputs["attention_mask"].to(decoder.device)

    with torch.no_grad():
        direct_output = decoder.target_model.generate(
            input_ids_direct,
            attention_mask=attention_mask_direct,
            max_new_tokens=max_tokens_check,
            do_sample=False,
            pad_token_id=decoder.target_tokenizer.pad_token_id,
        )
    direct_ids = direct_output[0, input_ids_direct.shape[1]:]

    spec_list   = spec_ids.tolist()
    direct_list = direct_ids.tolist()

    if spec_list == direct_list:
        print(f"  PASS: token sequences are identical ({len(spec_list)} tokens)\n")
    else:
        all_passed = False
        print(f"  FAIL: token sequences differ!")
        print(f"    spec   ({len(spec_list)} tokens): {spec_list}")
        print(f"    direct ({len(direct_list)} tokens): {direct_list}")
        for j, (s, d) in enumerate(zip(spec_list, direct_list)):
            if s != d:
                print(f"    First mismatch at position {j}: "
                      f"spec={s!r} ({decoder.target_tokenizer.decode([s])!r}), "
                      f"direct={d!r} ({decoder.target_tokenizer.decode([d])!r})")
                break
        if len(spec_list) != len(direct_list):
            print(f"    Length mismatch: spec={len(spec_list)}, direct={len(direct_list)}")
        print()

print("=" * 60)
print("Overall result:", "Correstness Test PASSED" if all_passed else "SOME CHECKS FAILED")

=== Correctness Check: Speculative Decoding vs. Direct Target Generation ===

Prompt 1: 'The future of Artificial Intelligence is'
Generated 50 tokens in 0.64 seconds
Tokens per second: 77.91
Draft token acceptance rate: 75.41%
  PASS: token sequences are identical (50 tokens)

Prompt 2: 'Write a short story about a robot learning to feel emotions:'
Generated 50 tokens in 0.54 seconds
Tokens per second: 92.98
Draft token acceptance rate: 87.04%
  PASS: token sequences are identical (50 tokens)

Prompt 3: "Write the lyrics to the song 'Happy Birthday'."
Generated 50 tokens in 0.60 seconds
Tokens per second: 83.29
Draft token acceptance rate: 78.33%
  PASS: token sequences are identical (50 tokens)

Overall result: Correstness Test PASSED


Please upload `Assignment.ipynb` to gradescope. This assignment is not automatically graded, and you will not receive immediate feedback upon submission. You can submit multiple times, but only your final submission will be graded.

Grading Criteria:

If correctness check cannot pass -> 0

Correctness check can pass, we test your implementation on Colab T4 GPU.

If the performance improvement is greater than 1.2x for all prompts -> 100

If the performance improvement is greater than 1.1x for all prompts -> 75

If the performance improvement is greater than 1.0x for all prompts -> 50